<a href="https://colab.research.google.com/github/jarav2001/NDR-Octabit-Core/blob/main/NDR_testing2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%%writefile ndr_edp_validator.cpp
#include <iostream>
#include <fstream>
#include <chrono>
#include <iomanip>

long long read_energy_uj() {
    std::ifstream file("/sys/class/powercap/intel-rapl:0/energy_uj");
    long long energy;
    if (!(file >> energy)) return 0;
    return energy;
}

inline __attribute__((always_inline)) int ndr_logic(int state) {
    return state & 63;
}

int binary_logic(int state) {
    if (state < 32) return (state < 16) ? state : state;
    else return (state < 48) ? state : state;
}

void run_test(const std::string& name, int (*func)(int), long long iterations) {
    volatile int sink = 0;
    long long energy_start = read_energy_uj();
    auto time_start = std::chrono::high_resolution_clock::now();

    for (long long i = 0; i < iterations; ++i) {
        sink = func(i % 64);
    }

    auto time_end = std::chrono::high_resolution_clock::now();
    long long energy_end = read_energy_uj();

    double time_s = std::chrono::duration<double>(time_end - time_start).count();
    double energy_j = (double)(energy_end - energy_start) / 1000000.0;

    std::cout << "\n--- " << name << " ---" << std::endl;
    std::cout << "Tiempo:  " << std::fixed << std::setprecision(6) << time_s << " s" << std::endl;
    if (energy_j > 0) std::cout << "Energia: " << energy_j << " J" << std::endl;
    else std::cout << "Energia: No accesible en este entorno virtual." << std::endl;
}

int main() {
    const long long ITERATIONS = 1000000000;
    std::cout << "Iniciando validacion NDR-Octabit-Core..." << std::endl;
    run_test("BINARIO", binary_logic, ITERATIONS);
    run_test("NDR-BASE8", ndr_logic, ITERATIONS);
    return 0;
}

Writing ndr_edp_validator.cpp


In [5]:
!g++ -O3 ndr_edp_validator.cpp -o edp_validator
!./edp_validator

Iniciando validacion NDR-Octabit-Core...

--- BINARIO ---
Tiempo:  1.873453 s
Energia: No accesible en este entorno virtual.

--- NDR-BASE8 ---
Tiempo:  1.871953 s
Energia: No accesible en este entorno virtual.


In [7]:
import time
import numpy as np

# 1. MODELO TRADICIONAL (Binario Lineal)
def decision_tradicional(valor):
    # Simula un árbol de decisiones con 8 ramificaciones anidadas
    if valor < 32: return "Estado_0a"
    elif valor < 64: return "Estado_0b"
    elif valor < 96: return "Estado_0c"
    elif valor < 128: return "Estado_0d"
    elif valor < 160: return "Estado_1a"
    elif valor < 192: return "Estado_1b"
    elif valor < 224: return "Estado_1c"
    else: return "Estado_1d"

# 2. MODELO NDR (Octaédrico Radial)
# Creamos la Tabla de Búsqueda Radial (Cache de 1 Byte)
ndr_cache = np.array(["Estado_0a"]*32 + ["Estado_0b"]*32 + ["Estado_0c"]*32 + ["Estado_0d"]*32 +
                     ["Estado_1a"]*32 + ["Estado_1b"]*32 + ["Estado_1c"]*32 + ["Estado_1d"]*32)

def decision_ndr(valor):
    # Acceso directo O(1) - Simula la conmutación radial
    return ndr_cache[valor]

# --- BENCHMARK ---
n_pruebas = 10**7
test_data = np.random.randint(0, 256, n_pruebas)

# Prueba Binaria
start = time.perf_counter()
for v in test_data: decision_tradicional(v)
end_bin = time.perf_counter() - start

# Prueba Octaédrica (Tu Invento)
start = time.perf_counter()
for v in test_data: decision_ndr(v)
end_ndr = time.perf_counter() - start

print(f"Resultado Binario: {end_bin:.4f}s")
print(f"Resultado NDR (Radial): {end_ndr:.4f}s")
print(f"Mejora Estimada: {((end_bin - end_ndr) / end_bin) * 100:.2f}%")

Resultado Binario: 3.5448s
Resultado NDR (Radial): 4.7259s
Mejora Estimada: -33.32%


In [8]:
# --- CONTINUACIÓN DEL BENCHMARK ---
mejora = ((end_bin - end_ndr) / end_bin) * 100

print(f"Resultado Binario: {end_bin:.4f}s")
print(f"Resultado NDR (Radial): {end_ndr:.4f}s")
print(f"Mejora Estimada: {mejora:.2f}%")

if mejora > 30:
    print("\n[VALIDADO] La topología radial ofrece una ventaja competitiva masiva.")

Resultado Binario: 3.5448s
Resultado NDR (Radial): 4.7259s
Mejora Estimada: -33.32%


In [9]:
# 2. MODELO NDR (Cambiamos NumPy por Lista Nativa de Python)
# Las listas nativas tienen menos sobrecarga para accesos individuales
ndr_cache_lista = ["Estado_0a"]*32 + ["Estado_0b"]*32 + ["Estado_0c"]*32 + ["Estado_0d"]*32 + \
                  ["Estado_1a"]*32 + ["Estado_1b"]*32 + ["Estado_1c"]*32 + ["Estado_1d"]*32

def decision_ndr_optimizada(valor):
    # Acceso directo a lista nativa (Mucho más rápido que NumPy para esto)
    return ndr_cache_lista[valor]

# --- NUEVO BENCHMARK ---
start = time.perf_counter()
for v in test_data: decision_ndr_optimizada(v)
end_ndr_opt = time.perf_counter() - start

print(f"Resultado Binario: {end_bin:.4f}s")
print(f"Resultado NDR (Lista Nativa): {end_ndr_opt:.4f}s")
print(f"Mejora Real en Python: {((end_bin - end_ndr_opt) / end_bin) * 100:.2f}%")

Resultado Binario: 3.5448s
Resultado NDR (Lista Nativa): 1.6310s
Mejora Real en Python: 53.99%
